In [1]:
!pip install -U bitsandbytes>=0.46.1 peft

In [2]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

hf_token = secrets.get_secret("HF_TOKEN")

os.environ["HF_TOKEN"] = hf_token

In [3]:
# ReAct Inference
# Models:
#   1. garam-icecream/phase2-toolalpaca-sft-phi2
#   2. souhhmm/phase2-toolalpaca-sft-qwen2_5-7b
#   3. souhhmm/phase2-toolalpaca-dpo-qwen2_5-7b

import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"


import ast
import importlib
import io
import json
import re
from contextlib import redirect_stdout
from datetime import datetime
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
import torch
from peft import PeftModel
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    StoppingCriteria,
    StoppingCriteriaList,
)

# HF Token
hf_token = None
try:
    kaggle_secrets_mod = importlib.import_module("kaggle_secrets")
    user_secrets_client = kaggle_secrets_mod.UserSecretsClient()
    hf_token = user_secrets_client.get_secret("HF_TOKEN")
    print("Loaded HF token from Kaggle secrets")
except Exception:
    hf_token = os.environ.get("HF_TOKEN")
    if hf_token:
        print("Loaded HF token from environment variable HF_TOKEN")

print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {props.name} | {props.total_memory / 1e9:.1f} GB")

# Config
CSV_PATH = "/kaggle/input/datasets/sohamkalburgi/agent-trajectories-2k/sales_data.csv"

MODELS = {
    # "phi2-sft": {
    #     "base":    "microsoft/phi-2",
    #     "adapter": "garam-icecream/phase2-toolalpaca-sft-phi2",
    # },
    # "qwen-sft": {
    #     "base":    "Qwen/Qwen2.5-7B-Instruct",
    #     "adapter": "souhhmm/phase2-toolalpaca-sft-qwen2_5-7b",
    # },
    "qwen-dpo": {
        "base":    "Qwen/Qwen2.5-7B-Instruct",
        "adapter": "souhhmm/phase2-toolalpaca-dpo-qwen2_5-7b",
    },
}

test_queries = [
    "What is the total revenue in 2022?",
    "What is the average profit by region?",
    "What are top 3 cities by revenue in 2022?",
]


# Stopping Criteria

class StopOnObservationCriteria(StoppingCriteria):
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.observation_tokens = tokenizer.encode("Observation:", add_special_tokens=False)

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor, **kwargs) -> bool:
        if len(input_ids[0]) >= len(self.observation_tokens):
            last_tokens = input_ids[0][-len(self.observation_tokens):].tolist()
            if last_tokens == self.observation_tokens:
                return True
        return False


class StopOnFinalAnswerCriteria(StoppingCriteria):
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.final_answer_tokens = tokenizer.encode("Final Answer:", add_special_tokens=False)

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor, **kwargs) -> bool:
        if len(input_ids[0]) >= len(self.final_answer_tokens):
            last_tokens = input_ids[0][-len(self.final_answer_tokens):].tolist()
            if last_tokens == self.final_answer_tokens:
                return True
        return False


# Parsing Utilities

def sanitize_json_string(s: str) -> str:
    s = re.sub(r'```json\n?', '', s)
    s = re.sub(r'```\n?', '', s)
    s = re.sub(r',\s*}', '}', s)
    s = re.sub(r',\s*]', ']', s)
    return s.strip()


def extract_action_name(text: str) -> Optional[str]:
    match = re.search(r'Action:\s*([a-zA-Z_][a-zA-Z0-9_]*)', text)
    if match:
        return match.group(1)
    return None


def extract_action_input(text: str) -> Optional[Dict[str, Any]]:
    match = re.search(r'Action\s+Input:\s*({.*?})', text, re.DOTALL)
    if match:
        json_str = match.group(1)
        brace_count = 0
        for i, char in enumerate(json_str):
            if char == '{':
                brace_count += 1
            elif char == '}':
                brace_count -= 1
                if brace_count == 0:
                    json_str = json_str[:i+1]
                    break
        json_str = sanitize_json_string(json_str)
        try:
            return json.loads(json_str)
        except json.JSONDecodeError:
            try:
                return ast.literal_eval(json_str)
            except Exception:
                return None

    shorthand = re.search(r'Action:\s*[a-zA-Z_][a-zA-Z0-9_]*\((.*?)\)', text, re.DOTALL)
    if shorthand:
        arg_text = shorthand.group(1).strip()
        if not arg_text:
            return {}
        parsed: Dict[str, Any] = {}
        for piece in re.split(r'\s*,\s*', arg_text):
            if '=' not in piece:
                continue
            key, value = piece.split('=', 1)
            key = key.strip()
            value = value.strip().strip('"').strip("'")
            if value.isdigit():
                parsed[key] = int(value)
            else:
                try:
                    parsed[key] = float(value)
                except ValueError:
                    parsed[key] = value
        return parsed
    return None


def extract_thought(text: str) -> Optional[str]:
    match = re.search(r'Thought:\s*(.+?)(?=\n|Action:|$)', text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return None


# Tool Executor

class ToolExecutor:
    def __init__(self, csv_path: str = "./sales_data.csv"):
        try:
            self.base_df = pd.read_csv(csv_path)
            print(f"Loaded sales data from {csv_path}")
            print(f"Shape: {self.base_df.shape}, Columns: {list(self.base_df.columns)}")
        except FileNotFoundError:
            raise FileNotFoundError(
                f"Could not find sales_data.csv at: {csv_path}. Please update the csv_path parameter."
            )
        self.current_df = self.base_df.copy()
        self.execution_history = []
        self.last_result = None
        self.last_sort_column = None

    def available_columns(self) -> List[str]:
        return list(self.base_df.columns)

    def is_grouped_view(self, group_col: Optional[str]) -> bool:
        if not group_col or group_col not in self.current_df.columns:
            return False
        return self.current_df[group_col].nunique(dropna=False) == len(self.current_df)

    def reset(self) -> Dict[str, Any]:
        self.current_df = self.base_df.copy()
        self.last_sort_column = None
        result = {"operation": "reset", "row_count": len(self.current_df), "status": "success"}
        self.last_result = result
        return result

    def filter_data(self, column: str = None, value: Any = None, **kwargs) -> Dict[str, Any]:
        try:
            if column is None or value is None:
                return {"error": "column and value required", "status": "failed"}
            if column not in self.current_df.columns:
                return {"error": f"Unknown column: {column}", "status": "failed"}
            self.current_df = self.current_df[self.current_df[column] == value].copy()
            result = {
                "operation": "filter",
                "parameters": {"column": column, "value": value},
                "row_count": len(self.current_df),
                "sample_rows": self.current_df.head(3).to_dict("records"),
                "status": "success",
            }
            self.execution_history.append(result)
            self.last_result = result
            return result
        except Exception as e:
            return {"error": str(e), "status": "failed", "operation": "filter_data"}

    def group_by(self, column: str = None, aggregate_column: str = None,
                 aggregate_function: str = "sum", **kwargs) -> Dict[str, Any]:
        try:
            if column is None:
                return {"error": "column required", "status": "failed"}
            if column not in self.current_df.columns:
                return {"error": f"Unknown column: {column}", "status": "failed"}
            agg_fn = kwargs.get("aggregation", aggregate_function)
            if agg_fn in {"avg", "average"}:
                agg_fn = "mean"
            if aggregate_column:
                if aggregate_column not in self.current_df.columns:
                    return {"error": f"Unknown aggregate column: {aggregate_column}", "status": "failed"}
                if aggregate_column == column:
                    return {"error": "aggregate_column must be different from group column",
                            "status": "failed", "operation": "group_by"}
                grouped = self.current_df.groupby(column)[aggregate_column].agg(agg_fn)
                grouped_df = grouped.reset_index(name=aggregate_column)
                self.current_df = grouped_df.copy()
                self.last_sort_column = aggregate_column
                result = {
                    "operation": "group_by",
                    "parameters": {"column": column, "aggregate_column": aggregate_column, "function": agg_fn},
                    "results": grouped.head(50).to_dict(),
                    "groups_count": len(grouped),
                    "status": "success",
                }
            else:
                grouped = self.current_df.groupby(column).size()
                result = {
                    "operation": "group_by",
                    "parameters": {"column": column},
                    "group_counts": grouped.to_dict(),
                    "groups_count": len(grouped),
                    "status": "success",
                }
            self.execution_history.append(result)
            self.last_result = result
            return result
        except Exception as e:
            return {"error": str(e), "status": "failed", "operation": "group_by"}

    def aggregate(self, column: str = None, aggregate_function: str = "sum", **kwargs) -> Dict[str, Any]:
        try:
            if column is None:
                return {"error": "column required", "status": "failed"}
            if column not in self.current_df.columns:
                return {"error": f"Unknown column: {column}", "status": "failed"}
            agg_fn = kwargs.get("aggregation", aggregate_function)
            if agg_fn in {"avg", "average"}:
                agg_fn = "mean"
            if agg_fn == "sum":
                result_value = self.current_df[column].sum()
            elif agg_fn == "mean":
                result_value = self.current_df[column].mean()
            elif agg_fn == "count":
                result_value = self.current_df[column].count()
            elif agg_fn == "max":
                result_value = self.current_df[column].max()
            elif agg_fn == "min":
                result_value = self.current_df[column].min()
            else:
                return {"error": f"Unknown function: {agg_fn}", "status": "failed"}
            result = {
                "operation": "aggregate",
                "parameters": {"column": column, "function": agg_fn},
                "result": float(result_value),
                "row_count": len(self.current_df),
                "status": "success",
            }
            self.execution_history.append(result)
            self.last_result = result
            return result
        except Exception as e:
            return {"error": str(e), "status": "failed", "operation": "aggregate"}

    def sort_by(self, column: str = None, order: str = "asc", **kwargs) -> Dict[str, Any]:
        try:
            if column is None:
                return {"error": "column required", "status": "failed"}
            if column not in self.current_df.columns:
                return {"error": f"Unknown column: {column}", "status": "failed"}
            ascending = order.lower() != "desc"
            self.current_df = self.current_df.sort_values(by=column, ascending=ascending).copy()
            self.last_sort_column = column
            result = {
                "operation": "sort_by",
                "parameters": {"column": column, "order": order},
                "sample_rows": self.current_df.head(5).to_dict("records"),
                "sorted_count": len(self.current_df),
                "status": "success",
            }
            self.execution_history.append(result)
            self.last_result = result
            return result
        except Exception as e:
            return {"error": str(e), "status": "failed", "operation": "sort_by"}

    def topk(self, column: str = None, k: int = 5, order: str = "desc", **kwargs) -> Dict[str, Any]:
        try:
            if column is None:
                column = kwargs.get("sort_by") or self.last_sort_column
            if column is None:
                return {"error": "column required", "status": "failed"}
            if column not in self.current_df.columns:
                return {"error": f"Unknown column: {column}", "status": "failed"}
            try:
                k = int(k)
            except Exception:
                k = 5
            k = max(1, k)
            ascending = order.lower() != "desc"
            if ascending:
                self.current_df = self.current_df.nsmallest(k, column)
            else:
                self.current_df = self.current_df.nlargest(k, column)
            result = {
                "operation": "topk",
                "parameters": {"column": column, "k": k, "order": order},
                "results": self.current_df.to_dict("records"),
                "count": len(self.current_df),
                "status": "success",
            }
            self.execution_history.append(result)
            self.last_result = result
            return result
        except Exception as e:
            return {"error": str(e), "status": "failed", "operation": "topk"}

    def top_k(self, **kwargs) -> Dict[str, Any]:
        return self.topk(**kwargs)

    def execute(self, tool_name: str, **kwargs) -> Dict[str, Any]:
        alias_map = {
            "topk":            ("topk",      {}),
            "top_k":           ("topk",      {}),
            "top-k":           ("topk",      {}),
            "aggregate_sum":   ("aggregate", {"aggregation": "sum"}),
            "aggregate_mean":  ("aggregate", {"aggregation": "mean"}),
            "aggregate_avg":   ("aggregate", {"aggregation": "mean"}),
            "aggregate_count": ("aggregate", {"aggregation": "count"}),
            "aggregate_max":   ("aggregate", {"aggregation": "max"}),
            "aggregate_min":   ("aggregate", {"aggregation": "min"}),
        }
        tool_name = (tool_name or "").strip().lower()
        mapped = alias_map.get(tool_name)
        if mapped is not None:
            tool_name = mapped[0]
            merged_kwargs = mapped[1].copy()
            merged_kwargs.update(kwargs)
            kwargs = merged_kwargs
        try:
            if hasattr(self, tool_name):
                return getattr(self, tool_name)(**kwargs)
            return {"error": f"Unknown tool: {tool_name}", "status": "failed"}
        except Exception as e:
            return {"error": str(e), "status": "failed", "tool": tool_name}


# ReAct Execution Engine

class ReActExecutionEngine:
    def __init__(self, model, tokenizer, tool_executor: ToolExecutor,
                 max_turns: int = 5, max_observation_length: int = 500):
        self.model = model
        self.tokenizer = tokenizer
        self.tool_executor = tool_executor
        self.max_turns = max_turns
        self.max_observation_length = max_observation_length
        self.conversation_history = []
        self.error_count = 0
        self.max_errors = 3

    def truncate_observation(self, observation: str) -> str:
        if len(observation) > self.max_observation_length:
            return observation[:self.max_observation_length] + "...[truncated]"
        return observation

    def format_system_prompt(self) -> str:
        return """You are an AI agent that solves data analysis tasks using available tools.

Available tools:
- filter_data
- group_by
- aggregate
- sort_by
- topk

Rules:
- Use tools only when needed.
- Use prior observations to decide the next action.
- Emit Final Answer only after enough evidence is available.

Response format:
Thought: [reasoning]
Action: [tool_name]
Action Input: {"param": value}

When done:
Final Answer: [natural-language answer]"""

    def _normalize_action_name(self, action_name: Optional[str]) -> Optional[str]:
        if not action_name:
            return None
        raw = action_name.strip()
        lowered = raw.lower().replace("-", "_").replace(" ", "_")
        if lowered in {"final_answer", "finalanswer"}:
            return "FINAL_ANSWER"
        alias_names = {
            "topk": "topk", "top_k": "topk",
            "aggregate_sum": "aggregate_sum", "aggregate_mean": "aggregate_mean",
            "aggregate_avg": "aggregate_avg", "aggregate_count": "aggregate_count",
            "aggregate_max": "aggregate_max", "aggregate_min": "aggregate_min",
            "group_by": "group_by", "filter_data": "filter_data",
            "sort_by": "sort_by", "aggregate": "aggregate",
        }
        return alias_names.get(lowered, lowered)

    def _normalize_token(self, token: Optional[str]) -> Optional[str]:
        if not token:
            return None
        t = token.strip().lower()
        if t.endswith("ies"):
            return t[:-3] + "y"
        if t.endswith("s") and not t.endswith("ss"):
            return t[:-1]
        return t

    def _column_by_token(self, token: Optional[str]) -> Optional[str]:
        if not token:
            return None
        norm = self._normalize_token(token)
        for c in self.tool_executor.available_columns():
            if c.lower() == norm:
                return c
        return None

    def _infer_metric_column(self, user_query: str) -> Optional[str]:
        q = user_query.lower()
        # "top N X by Y" -> Y is metric
        top_by = re.search(r"\btop\s+\d+\s+([a-zA-Z_]+)\s+by\s+([a-zA-Z_]+)", q)
        if top_by:
            col = self._column_by_token(top_by.group(2))
            if col:
                return col
        by_match = re.search(r"\bby\s+([a-zA-Z_]+)", q)
        if by_match:
            candidate = self._column_by_token(by_match.group(1))
            if candidate and candidate.lower() in {"revenue", "profit", "cost", "units_sold"}:
                return candidate
        for c in self.tool_executor.available_columns():
            if c.lower() in q:
                return c
        metric_hints = ["revenue", "profit", "cost", "units_sold", "units", "sales", "amount"]
        for hint in metric_hints:
            if hint in q:
                for c in self.tool_executor.available_columns():
                    lc = c.lower()
                    if hint == lc or hint in lc or lc in hint:
                        return c
        return None

    def _infer_group_column(self, user_query: str) -> Optional[str]:
        q = user_query.lower()
        # "top N X by Y" -> X is the grouping entity
        top_by = re.search(r"\btop\s+\d+\s+([a-zA-Z_]+)\s+by\s+([a-zA-Z_]+)", q)
        if top_by:
            col = self._column_by_token(top_by.group(1))
            if col:
                return col
        by_match = re.search(r"\bby\s+([a-zA-Z_]+)", q)
        if by_match:
            col = self._column_by_token(by_match.group(1))
            if col:
                return col
        for c in self.tool_executor.available_columns():
            if c.lower() in {"city", "region", "category", "product", "year", "month"} and c.lower() in q:
                return c
        return None

    def _extract_requested_k(self, user_query: str, default_k: int = 5) -> int:
        m = re.search(r"\btop\s+(\d+)\b", user_query.lower())
        if m:
            return max(1, int(m.group(1)))
        return default_k

    def _infer_agg_function(self, user_query: str, default_fn: str = "sum") -> str:
        q = user_query.lower()
        if "average" in q or "mean" in q or "avg" in q:
            return "mean"
        if "count" in q:
            return "count"
        if "maximum" in q or "max" in q:
            return "max"
        if "minimum" in q or "min" in q:
            return "min"
        if "total" in q or "sum" in q:
            return "sum"
        return default_fn

    def _is_aggregate_query(self, user_query: str) -> bool:
        return any(k in user_query.lower() for k in ["total", "sum", "average", "mean", "count", "max", "min"])

    def _is_topk_query(self, user_query: str) -> bool:
        return any(k in user_query.lower() for k in ["top", "highest", "lowest", "rank", "best", "worst"])

    def _is_grouped_aggregate_query(self, user_query: str) -> bool:
        return self._is_aggregate_query(user_query) and (" by " in user_query.lower())

    def _repair_action(self, user_query: str, action_name: str,
                       action_input: Optional[Dict[str, Any]]) -> Tuple[str, Dict[str, Any]]:
        action_name = self._normalize_action_name(action_name)
        action_input = action_input or {}
        if action_name == "FINAL_ANSWER":
            return action_name, action_input

        by_col = self._infer_group_column(user_query)
        metric_col = self._infer_metric_column(user_query)
        agg_fn = self._infer_agg_function(user_query)
        last_turn = self.conversation_history[-1] if self.conversation_history else None

        alias_names = {
            "aggregate_sum":   ("aggregate", "sum"),
            "aggregate_mean":  ("aggregate", "mean"),
            "aggregate_avg":   ("aggregate", "mean"),
            "aggregate_count": ("aggregate", "count"),
            "aggregate_max":   ("aggregate", "max"),
            "aggregate_min":   ("aggregate", "min"),
        }
        if action_name in alias_names:
            action_name, inferred_fn = alias_names[action_name]
            action_input.setdefault("aggregation", inferred_fn)

        if action_name == "group_by" and self._is_aggregate_query(user_query) and not self._is_grouped_aggregate_query(user_query):
            return "aggregate", {"column": metric_col, "aggregation": agg_fn}

        if action_name == "filter_data":
            if "column" in action_input and "value" not in action_input and by_col and action_input.get("column") == by_col:
                return "group_by", {"column": by_col}
            if self._is_topk_query(user_query) and by_col and metric_col and last_turn:
                same_filter = (
                    last_turn.get("action") == "filter_data"
                    and last_turn.get("action_input", {}).get("column") == action_input.get("column")
                    and last_turn.get("action_input", {}).get("value") == action_input.get("value")
                )
                if same_filter:
                    if not self.tool_executor.is_grouped_view(by_col):
                        return "group_by", {"column": by_col, "aggregate_column": metric_col, "aggregate_function": "sum"}
                    return "topk", {"column": metric_col, "k": self._extract_requested_k(user_query, default_k=5), "order": "desc"}

        if action_name == "group_by":
            if by_col:
                action_input["column"] = by_col
            if self._is_grouped_aggregate_query(user_query):
                action_input.setdefault("aggregate_column", metric_col)
                action_input.setdefault("aggregate_function", agg_fn)
            if self._is_topk_query(user_query) and by_col and metric_col:
                action_input.setdefault("aggregate_column", metric_col)
                action_input.setdefault("aggregate_function", "sum")

        if action_name == "aggregate":
            action_input.setdefault("column", metric_col)
            action_input.setdefault("aggregation", agg_fn)
            if self._is_grouped_aggregate_query(user_query) and by_col:
                return "group_by", {"column": by_col,
                                    "aggregate_column": action_input.get("column", metric_col),
                                    "aggregate_function": action_input.get("aggregation", agg_fn)}
            if self._is_topk_query(user_query) and by_col and metric_col:
                if self.tool_executor.is_grouped_view(by_col):
                    return "topk", {"column": action_input.get("column", metric_col),
                                    "k": self._extract_requested_k(user_query, default_k=5),
                                    "order": "desc"}
                return "group_by", {"column": by_col, "aggregate_column": metric_col, "aggregate_function": "sum"}

        if action_name in {"topk", "top_k"}:
            action_name = "topk"
            action_input.setdefault("k", self._extract_requested_k(user_query, default_k=5))
            action_input.setdefault("order", "desc")
            action_input.setdefault("column", metric_col or getattr(self.tool_executor, "last_sort_column", None))
            if self._is_topk_query(user_query) and by_col and metric_col:
                if not self.tool_executor.is_grouped_view(by_col):
                    return "group_by", {"column": by_col, "aggregate_column": metric_col, "aggregate_function": "sum"}

        if action_name == "sort_by":
            action_input.setdefault("column", metric_col)
            action_input.setdefault("order", "desc")
            if self._is_topk_query(user_query) and by_col and metric_col:
                if not self.tool_executor.is_grouped_view(by_col):
                    return "group_by", {"column": by_col, "aggregate_column": metric_col, "aggregate_function": "sum"}

        action_input = {k: v for k, v in action_input.items() if v is not None}
        return action_name, action_input

    def _format_aggregate_answer(self, aggregate_result: Dict[str, Any]) -> str:
        value = aggregate_result.get("result")
        params = aggregate_result.get("parameters", {})
        column = params.get("column", "value")
        fn = params.get("function", "sum")
        value_text = f"{value:,.2f}" if isinstance(value, float) else str(value)
        return f"Based on the analysis, the {fn} of {column} is {value_text}."

    def _format_topk_answer(self, user_query: str, topk_result: Dict[str, Any]) -> str:
        rows = topk_result.get("results", [])
        params = topk_result.get("parameters", {})
        metric_col = params.get("column", self._infer_metric_column(user_query) or "value")
        requested_k = self._extract_requested_k(user_query, default_k=params.get("k", 5))
        if not rows:
            return "I could not find matching results for this query."
        if isinstance(rows, dict):
            group_col = params.get("column", "group")
            items = sorted(rows.items(), key=lambda x: x[1], reverse=True)
            rows = [{group_col: k, metric_col: v} for k, v in items]
        rows = rows[:requested_k]
        lines = []
        for i, row in enumerate(rows, 1):
            entity = (row.get("city") or row.get("region") or row.get("product")
                      or row.get("category") or row.get("year") or row.get("month"))
            metric = row.get(metric_col, "N/A")
            if isinstance(metric, float):
                metric = f"{metric:,.2f}"
            if entity is None:
                lines.append(f"{i}. {metric_col}: {metric}")
            else:
                lines.append(f"{i}. {entity} ({metric_col}: {metric})")
        return f"Top {requested_k} results by {metric_col}:\n" + "\n".join(lines)

    def _format_groupby_answer(self, groupby_result: Dict[str, Any]) -> str:
        params = groupby_result.get("parameters", {})
        group_col = params.get("column", "group")
        if "results" in groupby_result:
            items = sorted(groupby_result["results"].items(), key=lambda x: x[1], reverse=True)
            lines = []
            for k, v in items[:10]:
                if isinstance(v, float):
                    v = f"{v:,.2f}"
                lines.append(f"- {k}: {v}")
            return f"Grouped summary by {group_col}:\n" + "\n".join(lines)
        if "group_counts" in groupby_result:
            items = sorted(groupby_result["group_counts"].items(), key=lambda x: x[1], reverse=True)
            lines = [f"- {k}: {v} rows" for k, v in items[:10]]
            return f"Row distribution by {group_col}:\n" + "\n".join(lines)
        return "I could not find grouped results to summarize."

    def _summarize_observation(self, result: Dict[str, Any]) -> str:
        if not isinstance(result, dict):
            return "Tool returned an unstructured response."
        if result.get("status") == "failed":
            op = result.get("operation") or result.get("tool") or "tool"
            return f"{op} failed: {result.get('error', 'unknown error')}."
        op = result.get("operation", "tool")
        if op == "filter":
            p = result.get("parameters", {})
            return f"Filtered {p.get('column')} = {p.get('value')}; {result.get('row_count')} rows remain."
        if op == "aggregate":
            p = result.get("parameters", {})
            val = result.get("result")
            if isinstance(val, float):
                val = f"{val:,.2f}"
            return f"{p.get('function')}({p.get('column')}) = {val}."
        if op == "group_by":
            p = result.get("parameters", {})
            if "aggregate_column" in p:
                return f"Grouped by {p.get('column')} and computed {p.get('function')}({p.get('aggregate_column')})."
            return f"Grouped by {p.get('column')} into {result.get('groups_count')} groups."
        if op == "sort_by":
            p = result.get("parameters", {})
            return f"Sorted by {p.get('column')} in {p.get('order')} order."
        if op == "topk":
            p = result.get("parameters", {})
            return f"Selected top {p.get('k')} by {p.get('column')}."
        return f"{op} completed successfully."

    def _format_final_answer_from_result(self, user_query: str, result: Dict[str, Any]) -> Optional[str]:
        if not isinstance(result, dict) or result.get("status") != "success":
            return None
        op = result.get("operation")
        if op == "aggregate":
            if self._is_topk_query(user_query):
                return None
            return self._format_aggregate_answer(result)
        if op == "group_by":
            if self._is_topk_query(user_query) and "results" in result:
                params = result.get("parameters", {})
                group_col = params.get("column", "group")
                metric_col = params.get("aggregate_column", self._infer_metric_column(user_query) or "value")
                grouped_items = sorted(result["results"].items(), key=lambda x: x[1], reverse=True)
                rows = [{group_col: k, metric_col: v} for k, v in grouped_items]
                synthetic = {
                    "operation": "topk",
                    "parameters": {"column": metric_col, "k": self._extract_requested_k(user_query, default_k=5)},
                    "results": rows,
                    "status": "success",
                }
                return self._format_topk_answer(user_query, synthetic)
            return self._format_groupby_answer(result)
        if op == "topk":
            return self._format_topk_answer(user_query, result)
        if op == "sort_by":
            sample_rows = result.get("sample_rows", [])
            synthetic = {
                "operation": "topk",
                "parameters": {"column": result.get("parameters", {}).get("column",
                               self._infer_metric_column(user_query) or "value"),
                               "k": self._extract_requested_k(user_query, default_k=5)},
                "results": sample_rows,
                "status": "success",
            }
            return self._format_topk_answer(user_query, synthetic)
        return None

    def generate_action(self, user_input: str, system_prompt: str) -> Tuple[str, Optional[str], Optional[Dict]]:
        full_prompt = f"""{system_prompt}\n\nUser Query: {user_input}\n\nThought:"""
        for turn in self.conversation_history:
            full_prompt += (
                f"\n{turn['thought']}\nAction: {turn['action']}\n"
                f"Action Input: {json.dumps(turn['action_input'])}\n"
                f"Observation: {turn['observation']}\n\nThought:"
            )

        inputs = self.tokenizer(full_prompt, return_tensors="pt", padding=True, truncation=True)
        inputs = {k: v.to(self.model.device) for k, v in inputs.items()}

        stopping_criteria = StoppingCriteriaList(
            [StopOnObservationCriteria(self.tokenizer), StopOnFinalAnswerCriteria(self.tokenizer)]
        )

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=256,
                temperature=0.7,
                top_p=0.95,
                do_sample=True,
                stopping_criteria=stopping_criteria,
                pad_token_id=self.tokenizer.eos_token_id,
            )

        generated_text = self.tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
        )

        if "Final Answer:" in generated_text:
            final_answer = generated_text.split("Final Answer:", 1)[1].strip()
            return generated_text, "FINAL_ANSWER", {"answer": final_answer}

        action_name = extract_action_name(generated_text)
        action_input = extract_action_input(generated_text)
        action_name = self._normalize_action_name(action_name)
        if action_name == "FINAL_ANSWER":
            return generated_text, "FINAL_ANSWER", action_input or {"answer": ""}
        return generated_text, action_name, action_input

    def execute_action(self, action_name: str, action_input: Dict) -> Dict[str, Any]:
        if not action_name:
            result = {"status": "failed", "error": "Invalid action format", "tool": "unknown"}
            return {"observation": self._summarize_observation(result), "result": result}
        if action_input is None:
            result = {"status": "failed", "error": "Invalid action input format", "tool": action_name}
            return {"observation": self._summarize_observation(result), "result": result}
        try:
            result = self.tool_executor.execute(action_name, **action_input)
            observation_text = self.truncate_observation(self._summarize_observation(result))
            return {"observation": observation_text, "result": result}
        except Exception as e:
            result = {"status": "failed", "error": str(e), "tool": action_name}
            return {"observation": self._summarize_observation(result), "result": result}

    def run(self, user_query: str) -> Dict[str, Any]:
        self.conversation_history = []
        self.error_count = 0
        if hasattr(self.tool_executor, "reset"):
            self.tool_executor.reset()

        system_prompt = self.format_system_prompt()

        for turn_number in range(self.max_turns):
            print(f"\n--- Turn {turn_number + 1}/{self.max_turns} ---")
            generated_text, action_name, action_input = self.generate_action(user_query, system_prompt)
            print(f"Generated: {generated_text[:200]}...")

            if action_name == "FINAL_ANSWER":
                last_result = getattr(self.tool_executor, "last_result", None)
                formatted = self._format_final_answer_from_result(user_query, last_result)
                if formatted:
                    print(f"Final Answer (formatted): {formatted}")
                    return {"status": "success", "final_answer": formatted,
                            "turns": len(self.conversation_history),
                            "actions": [h["action"] for h in self.conversation_history]}

            if not action_name:
                self.error_count += 1
                print("Failed to parse action. Retrying.")
                if self.error_count >= self.max_errors:
                    break
                continue

            action_name, action_input = self._repair_action(user_query, action_name, action_input)
            if action_name == "FINAL_ANSWER":
                last_result = getattr(self.tool_executor, "last_result", None)
                formatted = self._format_final_answer_from_result(user_query, last_result)
                if formatted:
                    return {"status": "success", "final_answer": formatted,
                            "turns": len(self.conversation_history),
                            "actions": [h["action"] for h in self.conversation_history]}
                continue

            execution = self.execute_action(action_name, action_input)
            observation = execution["observation"]
            raw_result = execution["result"]

            print(f"Action: {action_name}")
            print(f"Observation: {observation[:200]}...")

            self.conversation_history.append({
                "thought": extract_thought(generated_text) or "[thought]",
                "action": action_name,
                "action_input": action_input or {},
                "observation": observation,
                "raw_result": raw_result,
            })

            last_result = getattr(self.tool_executor, "last_result", None)
            if isinstance(last_result, dict) and last_result.get("status") == "success":
                op = last_result.get("operation")
                if self._is_grouped_aggregate_query(user_query):
                    if op == "group_by" and "aggregate_column" in last_result.get("parameters", {}):
                        final_answer = self._format_groupby_answer(last_result)
                        print(f"Final Answer (auto): {final_answer}")
                        return {"status": "success", "final_answer": final_answer,
                                "turns": len(self.conversation_history),
                                "actions": [h["action"] for h in self.conversation_history]}
                elif self._is_aggregate_query(user_query):
                    if op == "aggregate":
                        final_answer = self._format_aggregate_answer(last_result)
                        print(f"Final Answer (auto): {final_answer}")
                        return {"status": "success", "final_answer": final_answer,
                                "turns": len(self.conversation_history),
                                "actions": [h["action"] for h in self.conversation_history]}
                if self._is_topk_query(user_query) and op in {"topk", "group_by"}:
                    if op == "group_by":
                        final_answer = self._format_final_answer_from_result(user_query, last_result)
                    else:
                        final_answer = self._format_topk_answer(user_query, last_result)
                    print(f"Final Answer (auto): {final_answer}")
                    return {"status": "success", "final_answer": final_answer,
                            "turns": len(self.conversation_history),
                            "actions": [h["action"] for h in self.conversation_history]}

        last_result = getattr(self.tool_executor, "last_result", None)
        fallback = self._format_final_answer_from_result(user_query, last_result)
        if fallback:
            return {"status": "success", "final_answer": fallback,
                    "turns": len(self.conversation_history),
                    "actions": [h["action"] for h in self.conversation_history]}
        return {"status": "completed", "final_answer": None,
                "turns": len(self.conversation_history),
                "actions": [h["action"] for h in self.conversation_history],
                "error": "Max turns reached without final answer"}


print(" All classes defined")


# Model Loader

def load_finetuned_model(base_model_id: str, adapter_repo: str):
    supports_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    compute_dtype = torch.bfloat16 if supports_bf16 else torch.float16
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=compute_dtype,
    )

    print(f"  Loading tokenizer: {base_model_id}")
    ft_tokenizer = AutoTokenizer.from_pretrained(base_model_id, trust_remote_code=True, token=hf_token)
    if ft_tokenizer.pad_token is None:
        ft_tokenizer.add_special_tokens({"pad_token": "[PAD]"})
    ft_tokenizer.padding_side = "left"

    # Patch the config before model load so PhiConfig has pad_token_id set
    from transformers import AutoConfig
    config = AutoConfig.from_pretrained(base_model_id, trust_remote_code=True, token=hf_token)
    if not hasattr(config, "pad_token_id") or config.pad_token_id is None:
        config.pad_token_id = ft_tokenizer.pad_token_id

    print(f"  Loading base model: {base_model_id}")
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_id,
        config=config,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        max_memory={0: "14GiB", 1: "14GiB", "cpu": "48GiB"},
        token=hf_token,
    )

    base_model.resize_token_embeddings(len(ft_tokenizer))

    print(f"  Loading adapter: {adapter_repo}")
    ft_model = PeftModel.from_pretrained(base_model, adapter_repo, device_map="auto", token=hf_token)
    ft_model.eval()
    ft_model.config.use_cache = True
    print(f"   Loaded successfully")
    return ft_model, ft_tokenizer


def release_model(model, tokenizer):
    import gc
    del model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


# Run All Models

tool_executor = ToolExecutor(csv_path=CSV_PATH)
all_results: Dict[str, List[Dict]] = {}
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

for model_key, cfg in MODELS.items():
    print(f"\n{'='*60}")
    print(f"MODEL: {model_key}  ({cfg['adapter']})")
    if torch.cuda.is_available():
        free = torch.cuda.mem_get_info()[0] / 1e9
        print(f"GPU free before load: {free:.1f} GB")
    print("="*60)

    try:
        ft_model, ft_tokenizer = load_finetuned_model(cfg["base"], cfg["adapter"])
        engine = ReActExecutionEngine(
            model=ft_model,
            tokenizer=ft_tokenizer,
            tool_executor=tool_executor,
            max_turns=5,
            max_observation_length=600,
        )
    except Exception as e:
        print(f"  ✗ Failed to load: {e}")
        all_results[model_key] = [{"error": str(e)}]
        try: release_model(ft_model, ft_tokenizer)
        except Exception: pass
        continue

    log_buffer = io.StringIO()
    results = []

    with redirect_stdout(log_buffer):
        for i, query in enumerate(test_queries):
            print(f"\n\nQuery {i+1}: {query}")
            print("-" * 60)
            try:
                result = engine.run(query)
            except Exception as e:
                result = {"status": "error", "error": str(e), "final_answer": None}
            result["query"] = query
            result["turn_history"] = engine.conversation_history
            results.append(result)
            print(f"\nFinal Result:\n{json.dumps(result, indent=2)}")

    all_results[model_key] = results

    log_path = f"./execution_log_{model_key}_{timestamp}.txt"
    with open(log_path, "w") as f:
        f.write(log_buffer.getvalue())

    print(log_buffer.getvalue())
    print(f" Log saved to {log_path}")

    del engine
    release_model(ft_model, ft_tokenizer)
    if torch.cuda.is_available():
        free = torch.cuda.mem_get_info()[0] / 1e9
        print(f"GPU free after release: {free:.1f} GB")


# Save Results and Print Summary

output_file = f"./phase2_inference_results_{timestamp}.json"
with open(output_file, "w") as f:
    json.dump(all_results, f, indent=2)

print(f"\n\n{'='*70}")
print("RESULTS SUMMARY")
print("="*70)
for query in test_queries:
    print(f"\nQuery: {query}")
    for model_key in MODELS:
        res = next((r for r in all_results.get(model_key, []) if r.get("query") == query), {})
        ans  = res.get("final_answer") or res.get("error", "N/A")
        acts = res.get("actions", [])
        print(f"  [{model_key:<10}] {ans}")
        if acts:
            print(f"               actions: {' -> '.join(acts)}")

print(f"\n Results saved to {output_file}")

Loaded HF token from Kaggle secrets
GPU Available: True
  GPU 0: Tesla T4 | 15.6 GB
  GPU 1: Tesla T4 | 15.6 GB
 All classes defined
Loaded sales data from /kaggle/input/datasets/sohamkalburgi/agent-trajectories-2k/sales_data.csv
Shape: (10000, 11), Columns: ['date', 'year', 'month', 'city', 'region', 'product', 'category', 'revenue', 'units_sold', 'cost', 'profit']

MODEL: qwen-dpo  (souhhmm/phase2-toolalpaca-dpo-qwen2_5-7b)
GPU free before load: 15.5 GB
  Loading tokenizer: Qwen/Qwen2.5-7B-Instruct


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

  Loading adapter: souhhmm/phase2-toolalpaca-dpo-qwen2_5-7b


adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/323M [00:00<?, ?B/s]

   Loaded successfully


Query 1: What is the total revenue in 2022?
------------------------------------------------------------

--- Turn 1/5 ---
Generated:  No tool is needed for this step as we can directly filter the data for year 2022 and aggregate the revenue.

Action: filter_data
Action Input: {"column": "year", "value": 2022}...
Action: filter_data
Observation: Filtered year = 2022; 3304 rows remain....

--- Turn 2/5 ---
Generated:  The total revenue for 2022 can be found by aggregating the revenue column.
Action: aggregate
Action Input: {"column": "revenue", "aggregation": "sum"}...
Action: aggregate
Observation: sum(revenue) = 44,926,739.48....
Final Answer (auto): Based on the analysis, the sum of revenue is 44,926,739.48.

Final Result:
{
  "status": "success",
  "final_answer": "Based on the analysis, the sum of revenue is 44,926,739.48.",
  "turns": 2,
  "actions": [
    "filter_data",
    "aggregate"
  ],
  "query": "What is the total revenue in 2022?",
  "turn_history"